In [10]:
from llama_index.core import SimpleDirectoryReader

In [11]:
documents = SimpleDirectoryReader(input_files=[r"../data/63d00cdb-f5fb-4d86-9a8e-4cea460455f1.md"]).load_data()
text = documents[0].text
new_job_id = str(uuid.uuid4())
initial_state = PipelineState(
    job_id=new_job_id,
    raw_text=text,
    metadata=None,
    knowledge_base=None,
    validation_errors=[],
    current_stage="Initialized"
)

In [23]:
import base64
from io import BytesIO
from PIL import Image
import json

def base64_to_image(base64_string, output_filename="output_image.png"):
    if "," in base64_string:
        base64_string = base64_string.split(",")[1]
        
    image_data = base64.b64decode(base64_string)
    image_stream = BytesIO(image_data)
    image = Image.open(image_stream)
    image.save(output_filename)
    print(f"Success! Image saved as {output_filename}")

with open(r"../data/63d00cdb-f5fb-4d86-9a8e-4cea460455f1.json", 'r') as f:
    res = json.load(f)
    print(type(res))

for key, val in res.items():
    # print(key)
    base64_to_image(val, f"../data/images/{key}.jpg")


<class 'dict'>
Success! Image saved as ../data/images/page_1_image_2_v2.jpg
Success! Image saved as ../data/images/page_1_image_3_v2.jpg
Success! Image saved as ../data/images/page_1_image_4_v2.jpg
Success! Image saved as ../data/images/page_1_image_1_v2.jpg
Success! Image saved as ../data/images/page_3_image_1_v2.jpg
Success! Image saved as ../data/images/page_3_image_2_v2.jpg
Success! Image saved as ../data/images/page_3_image_5_v2.jpg
Success! Image saved as ../data/images/page_3_image_6_v2.jpg
Success! Image saved as ../data/images/page_3_image_7_v2.jpg
Success! Image saved as ../data/images/page_3_image_4_v2.jpg
Success! Image saved as ../data/images/page_3_image_3_v2.jpg
Success! Image saved as ../data/images/page_4_image_3_v2.jpg
Success! Image saved as ../data/images/page_4_image_1_v2.jpg
Success! Image saved as ../data/images/page_4_image_2_v2.jpg
Success! Image saved as ../data/images/page_6_image_1_v2.jpg
Success! Image saved as ../data/images/page_6_image_2_v2.jpg
Success! 

In [ ]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from core.config import GEMINI_API_KEY
import json

class ImageInfo(BaseModel):
    description: str = Field(description="A description of the image")
    table: bool = Field(description="Whether there is a table in the image")
    table_content: str = Field(description="The content of the table if there is one")

llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", api_key=GEMINI_API_KEY)
structured_llm = llm.with_structured_output(ImageInfo)

def extract_image_info(base64_data: str) -> dict:
    """
    Extracts structured information from an image given the following output schema:
    - description: A description of the image
    - table: Whether there is a table in the image
    - table_content: The content of the table if there is one

    Args:
        base64_data: The URL of the image to analyze

    Returns:
        A dictionary containing the description, table, and table content
    """

    
    response = structured_llm.invoke([
        SystemMessage(content="You are a helpful assistant that extracts structured information from an image."),
        HumanMessage(
            content=[
                {"type": "text", "text": "Analyze the image"},
                {
                    "type": "image_url",
                    "image_url" : {
                        "url": base64_data
                    }
                },
            ],
        )
    ])
    return response.model_dump()
output = ''
with open(r"../data/63d00cdb-f5fb-4d86-9a8e-4cea460455f1.json", 'r') as f:
    res = json.load(f)
    # print(type(res))
    output = extract_image_info(res["page_8_image_1_v2"])

In [9]:
output

{'description': "The image illustrates the phenomenon of atmospheric refraction, showing how light from a distant star bends as it passes through Earth's atmosphere. A 'Star' is depicted on the upper left, emitting a light ray that follows a curved 'Ray path' towards an observer on the lower right. An arrow on the left indicates that the 'Refractive index' is 'increasing' downwards, implying denser atmospheric layers closer to the ground. Due to this bending of light, the observer perceives the star at an 'Apparent star position' which is higher than its actual location, indicated by a dashed line extending straight from the observer's eye.",
 'table': False,
 'table_content': ''}